# 01 — Chuẩn bị MVTec AD trên Kaggle (v2)

Notebook của **giai đoạn 1**: kiểm tra dữ liệu, tạo split cố định, xuất báo cáo và ZIP. Chưa train model.

**Input:** `/kaggle/input/datasets/ipythonx/mvtec-ad` và bundle package.
**Output:** `/kaggle/working/visual-ad` cùng ZIP tải về.

Bản v2 chạy code package trong process Python mới sau khi cài dependency. Điều này tránh việc kernel Kaggle giữ module Pillow cũ trong RAM trong khi file package đã được nâng cấp.

## 1. Cài package và xác minh bundle

Sửa `BUNDLE_ROOT` nếu mount Kaggle khác. Checksum phát hiện file bundle thay đổi. `USE_INSTALLED_PACKAGE` chỉ phục vụ kiểm thử local.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

BUNDLE_ROOT = Path(os.environ.get("VAD_BUNDLE_ROOT", "/kaggle/input/visual-ad-phase1"))
USE_INSTALLED_PACKAGE = os.environ.get("VAD_USE_INSTALLED_PACKAGE") == "1"
assert (3, 11) <= sys.version_info[:2] < (3, 13), "Package hỗ trợ Python 3.11 hoặc 3.12"

if not USE_INSTALLED_PACKAGE:
    provenance = json.loads((BUNDLE_ROOT / "bundle-manifest.json").read_text())
    for relative, expected in provenance["files"].items():
        path = (BUNDLE_ROOT / relative).resolve()
        assert path.is_relative_to(BUNDLE_ROOT.resolve()), "Path ngoài bundle"
        assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, f"Checksum sai: {relative}"
    wheels = list(BUNDLE_ROOT.glob("visual_ad_lab-*.whl"))
    assert len(wheels) == 1, "Bundle phải có đúng một wheel visual-ad-lab"
    subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                    str(BUNDLE_ROOT / "requirements-kaggle.lock")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps",
                    "--force-reinstall", str(wheels[0])], check=True)

# visual_ad chỉ được import trong process mới sau pip install.
probe = subprocess.run([sys.executable, "-m", "visual_ad", "environment"],
                       check=True, text=True, capture_output=True)
installed = json.loads(probe.stdout)
if not USE_INSTALLED_PACKAGE:
    assert installed["source_fingerprint"] == provenance["source_fingerprint"], "Source đang cài khác bundle"
print(json.dumps(installed, indent=2))

## 2. Cấu hình đường dẫn

Dataset input chỉ đọc. Output nằm ngoài dataset. Seed 42 và protocol normal 70/15/15, test 40/60 được khóa trong artifact.

In [ ]:
config_path = BUNDLE_ROOT / "configs" / "data.yaml"
DATA_ROOT = Path(os.environ.get("VAD_DATA_ROOT", "/kaggle/input/datasets/ipythonx/mvtec-ad"))
OUTPUT_ROOT = Path(os.environ.get("VAD_OUTPUT_ROOT", "/kaggle/working/visual-ad"))
SEED = int(os.environ.get("VAD_SEED", "42"))
print({"data_root": str(DATA_ROOT), "output_root": str(OUTPUT_ROOT), "seed": SEED,
       "config": str(config_path) if config_path.exists() else None})

## 3. Kiểm tra dữ liệu và tạo manifest

Lệnh decode ảnh, kiểm tra mask, duplicate, hash nội dung và chia từng category/defect type. Process mới nhận toàn bộ phiên bản Pillow vừa cài nên không bị trạng thái module trộn trong kernel.

In [ ]:
arguments = [sys.executable, "-m", "visual_ad", "data", "prepare",
             "--root", str(DATA_ROOT), "--output", str(OUTPUT_ROOT), "--seed", str(SEED)]
if config_path.exists():
    arguments += ["--config", str(config_path)]
subprocess.run(arguments, check=True)

## 4. Đọc báo cáo và kiểm tra trực quan

Mỗi hàng: ảnh gốc — ground-truth mask — overlay. Gallery chỉ dùng fit/selection; không hiển thị final_test.

In [ ]:
from IPython.display import Markdown, Image as DisplayImage, display

report = json.loads((OUTPUT_ROOT / "data-report.json").read_text())
print("Số ảnh:", report["sample_count"])
print("Số category:", report["category_count"])
print("Split fingerprint:", report["split_fingerprint"])
print("Cảnh báo:", report["warnings"])
display(Markdown((OUTPUT_ROOT / "data-report.md").read_text()))
display(DisplayImage(filename=str(OUTPUT_ROOT / "data-preview.png")))

## 5. Kiểm tra canonical sample

Adapter phải tạo RGB float32 `[C,H,W]` trong `[0,1]` và mask bool `[H,W]`. Kiểm tra chạy trong process mới.

In [ ]:
sample_script = """import json, sys
from visual_ad.data import load_manifest, load_sample, samples_for
manifest = load_manifest(sys.argv[2])
row = samples_for(manifest, manifest["categories"][0], "fit")[0]
sample = load_sample(sys.argv[1], row, 256)
assert not sample["mask"].any()
print(json.dumps({"image_shape": list(sample["image"].shape),
                  "image_dtype": str(sample["image"].dtype),
                  "mask_shape": list(sample["mask"].shape),
                  "mask_dtype": str(sample["mask"].dtype),
                  "original_size": sample["original_size"]}))
"""
sample_result = subprocess.run([sys.executable, "-c", sample_script, str(DATA_ROOT),
                                str(OUTPUT_ROOT / "split-manifest.json")],
                               check=True, text=True, capture_output=True)
print(json.dumps(json.loads(sample_result.stdout), indent=2))

## 6. Đóng gói và tải kết quả

ZIP chứa config, inventory, split manifest, trạng thái, report, CSV, preview và `checksums.json`; không chứa ảnh dataset. Tên file mang 12 ký tự đầu của split fingerprint.

In [ ]:
archive_result = subprocess.run([sys.executable, "-m", "visual_ad", "data", "archive",
                                 "--output", str(OUTPUT_ROOT)],
                                check=True, text=True, capture_output=True)
archive_info = json.loads(archive_result.stdout)
print(json.dumps(archive_info, ensure_ascii=False, indent=2))

from IPython.display import FileLink
display(FileLink(archive_info["archive"], result_html_prefix="Tải ZIP kết quả: "))

Sau khi tải ZIP, vẫn chọn **Save Version → Save & Run All** để lưu output trên Kaggle. Notebook này chỉ nghiệm thu dữ liệu; chưa chứng minh mô hình đã huấn luyện.